# Testscript for Master Thesis 

rm -rf .git#### Open ToDo's
-Clean tweets from nbsps etc.
-rewrite data frame creation and data collection into single functions

In [165]:
! rm -rf .git

In [84]:
# Setup
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


### SANDBOX & SETUP

In [109]:
# Chat GPT API Key 
# Currently Pushed... --> Change before making public 
from api_key import API_KEY as API_KEY

In [110]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 35 and 47. Only return the number"

# Test completion
completion = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     "content": return_number
    }
  ],
  temperature = 2 # default temp = 1  
)

print("Response: ", completion.choices[0].message.content);

Response:  43


### LABELLING MULTIPLE TWEETS IN A DF

#### Load the Tweets from the Kern et al. (2023) paper

In [123]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Replace these with the actual paths or filenames
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

# Load the datasets
train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

In [124]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet.id'].nunique()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
tweets_full.sort_values('tweet.id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000


,tweet.id,id,version,batch.tweet,hate.speech,offensive.language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


#### Randomly Sample X tweets and create the result df out of them

In [146]:
import pandas as pd

# randomly sample 3 unique tweets
N_SAMPLES = 100

sampled_tweets = (
    tweets_full
    .drop_duplicates(subset='tweet_hashed')
    .sample(n=N_SAMPLES
            ,random_state=42 # no seed to get different cases during dev
           )
    ['tweet_hashed']
)

# tweets = [
#     "I live for two-bite brownies",
#     "@###### see bitch this is why you have no friends",
#     "I f*****g love this"
# ]

# Build a list of dicts: 5 rows (conditions A–E) for each of the 3 tweets
# CONDITIONS = ['C.OL', 'C.HS']
# CONDITIONS = ['A', 'B', 'C.OL', 'C.HS']
CONDITIONS = ['A', 'B', 'C.OL', 'C.HS']

rows = []
for tweet_id, text in enumerate(sampled_tweets, start=1):
    for condition in CONDITIONS:
        rows.append({
            "tweet_id": tweet_id,
            "tweet": text,
            "condition": condition
        })

# Create the DataFrame
N_RESPONSES = 3
dat = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    dat[f"response_{i}"] = ""
    

# Verify it has 15 rows
print(dat.shape)
dat.head(15)

(400, 6)


,tweet_id,tweet,condition,response_1,response_2,response_3
0,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,A,,,
1,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,B,,,
2,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,C.OL,,,
3,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,C.HS,,,
4,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,A,,,
5,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,B,,,
6,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,C.OL,,,
7,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,C.HS,,,
8,3,@###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot,A,,,
9,3,@###### shut the fuck up. You guys a mediocre washed basura juice. Sit down and win close games against raiders and jets faggot,B,,,


In [147]:
# Define the prompts

PROMPT_DICT = {
    # A: Both OL and HS at the same time, OL first
    "A":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or Hate Speech (HS). 
    1) Return *OL* if the tweet contains Offensive Language
    2) Return *HS* if the tweet contains Hate Speech
    If they contain both OL and HS return *OLHS*, if neither OL nor HS return *none*
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    # B: Both OL and HS at the same time, HS first
    "B":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or Offensive Language (OL). 
    1) Return *HS* if the tweet contains Hate Speech
    2) Return *OL* if the tweet contains Offensive language
    If they contain both HS and OL return *HSOL*, if neither HS nor OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    # C.OL: Only OL
    "C.OL":"""You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    Return *OL* if the tweet contains Offensive language
    If they do not contain OL return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""", 
    # C.HS: Only HS
    "C.HS":"""You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    Return *HS* if the tweet contains Hate Speech
    If they do not contain HS return *none*.
    Please return only the parts in between the *asterisks* (without asterisks), do not add any explanations.
    Here is the tweet: {tweet}""" 
    # IN BATCHES (of 50)
    # B_A: batches with Both OL and HS at the same time (OL first)
    
    
    # B_B: batches with Both OL and HS at the same time (HS first)
    
    
    # ,"X":"Say X", 
    # "Y":"Say Y"
}

In [161]:
%%time 

# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# Conditions A, B, C.HS, C.OL
CONDITIONS = ['A', 'B', 'C.HS', 'C.OL'] # this should be the same as above
# CONDITIONS = ['A']

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3


# Create results df
results = dat
results = results[results['condition'].isin(CONDITIONS)].reset_index(drop=True)
results

for row in range(0,results.shape[0]):
    # Option A) paste prompt and tweet dynamically 
    # prompt = PROMPT_DICT[row["condition"]].format(tweet=row["tweet"])
    
    # Option B) only use prompt for a specific condition
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt = PROMPT_DICT[condition].format(tweet=tweet_to_rate)
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=1,
            store=True,
        )
        label = resp.choices[0].message.content.strip()
        # print(prompt)
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"response_{response}"] = label

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results

Number of API Calls: 1200
CPU times: user 11.7 s, sys: 926 ms, total: 12.6 s
Wall time: 12min 42s


,tweet_id,tweet,condition,response_1,response_2,response_3
0,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,A,none,none,none
1,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,B,none,none,none
2,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,C.OL,OL,OL,OL
3,1,RT @###### Chile I'm taking greyhound everywhere from now on. Yo hoes and bout to fly me into no building or fly me over the Bermuda &#8230;,C.HS,none,none,none
4,2,@###### nicca u read what i typed u aint slo by a long shot dont play dumb now litebrite lol,A,OLHS,OLHS,OLHS
...,...,...,...,...,...,...
395,99,@###### ugly emo trash go cut,C.HS,HS,HS,HS
396,100,@###### #Sequester Den the fat colored girl chased me down day street while she swingin da table wit one arm... I wants my pay!!!,A,OLHS,OLHS,OLHS
397,100,@###### #Sequester Den the fat colored girl chased me down day street while she swingin da table wit one arm... I wants my pay!!!,B,HSOL,HS,HSOL
398,100,@###### #Sequester Den the fat colored girl chased me down day street while she swingin da table wit one arm... I wants my pay!!!,C.OL,OL,OL,OL


In [163]:
# save result file 
import os 
from datetime import date

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = "100t_4c_3r"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

# Notes 
1) The tweets can be processed in parts to either (e.g., the first 2k for all conditions)  
A) be able to split the task into free sub parts or   
B) to be able to better estimate the total costs  

2) Probably the Batch Calls have to be done in a separate code snippet

3) Costs: 
4o-mini : 
Input:          15ct  /1M Tokens
Cached Input:   7.5ct /1M Tokens (might be our case)
Output:         60ct  /1M Tokens


In [136]:
# Get Token Counts 
from utils import count_tokens
import statistics
# count tokens for gpt-3.5-turbo model (default in tiktoken package 
print(count_tokens(PROMPT_DICT['A'])) # 111 TOKENS for standard condition prompt 
print(statistics.fmean([count_tokens(x) for x in sampled_tweets])) # 15 Average Tweet Token length
# response has max 2 tokens

111
15.8
